In [ ]:
"""
==============================================================================
CONVERSÃO RNDS  CSV → Parquet
Usa DuckDB para leitura streaming diretamente do CSV → Parquet,
sem carregar o arquivo inteiro na RAM.
==============================================================================
"""
import duckdb
import os

# ---------------------------------------------------------------------------
# Configurações
# ---------------------------------------------------------------------------
CSV_PATH     = "base/tb_ra_202511241538.csv"
PARQUET_PATH = "base/RNDS.parquet"

# Memória máxima permitida ao DuckDB (ajuste conforme o servidor)
DUCKDB_MEMORY = "2GB"

# ---------------------------------------------------------------------------
# Conversão
# ---------------------------------------------------------------------------
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Arquivo não encontrado: {CSV_PATH}")

con = duckdb.connect(database=":memory:")
con.execute(f"SET memory_limit='{DUCKDB_MEMORY}';")
con.execute("SET threads TO 4;")

print(f"Convertendo: {CSV_PATH}  →  {PARQUET_PATH}")

con.execute(f"""
COPY (
    SELECT
        CAST(nu_cpf_paciente            AS VARCHAR) AS nu_cpf_paciente,
        CAST(nu_cns_paciente            AS VARCHAR) AS nu_cns_paciente,
        CAST(co_sigtap                  AS VARCHAR) AS co_sigtap,
        CAST(co_cbo                     AS VARCHAR) AS co_cbo,
        CAST(sg_uf_estab_executante     AS VARCHAR) AS sg_uf_estab_executante,
        CAST(co_municipio_estab_executante AS VARCHAR) AS co_municipio_estab_executante,
        CAST(co_cnes_estab_executante   AS VARCHAR) AS co_cnes_estab_executante,
        -- Datas no formato DD/MM/YYYY → DATE
        TRY_STRPTIME(TRIM(data_solicitacao), '%d/%m/%Y')::DATE AS data_solicitacao,
        TRY_STRPTIME(TRIM(data_autorizacao),  '%d/%m/%Y')::DATE AS data_autorizacao,
        TRY_STRPTIME(TRIM(data_execucao),     '%d/%m/%Y')::DATE AS data_execucao,
        -- Boolean: aceita 'true'/'false'/'1'/'0'
        CASE LOWER(TRIM(CAST(st_vida_paciente AS VARCHAR)))
            WHEN 'true'  THEN TRUE
            WHEN '1'     THEN TRUE
            WHEN 'false' THEN FALSE
            WHEN '0'     THEN FALSE
            ELSE NULL
        END AS st_vida_paciente,
        TRIM(CAST(st_solicitacao              AS VARCHAR)) AS st_solicitacao,
        TRIM(CAST(id_registro_sistema_origem  AS VARCHAR)) AS id_registro_sistema_origem,
        TRIM(CAST(ds_sistema_origem           AS VARCHAR)) AS ds_sistema_origem
    FROM read_csv(
        '{CSV_PATH}',
        delim        = ';',
        quote        = '"',
        header       = true,
        all_varchar  = true,   -- lê tudo como texto; as conversões são feitas acima
        ignore_errors= true,
        parallel     = true
    )
)
TO '{PARQUET_PATH}'
(FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 500000);
""")

con.close()
print(f"✅ Parquet gerado: {PARQUET_PATH}")


Conversão concluída: base/tb_ra_202511241538.parquet


In [ ]:
"""
==============================================================================
CONVERSÃO SIA  CSV → Parquet
==============================================================================
"""
import duckdb
import os

CSV_PATH     = "base/SIA.csv"
PARQUET_PATH = "base/SIA.parquet"
DUCKDB_MEMORY = "2GB"

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Arquivo não encontrado: {CSV_PATH}")

con = duckdb.connect(database=":memory:")
con.execute(f"SET memory_limit='{DUCKDB_MEMORY}';")
con.execute("SET threads TO 4;")

print(f"Convertendo: {CSV_PATH}  →  {PARQUET_PATH}")

con.execute(f"""
COPY (
    SELECT
        TRIM(CAST(CPF_PAC                   AS VARCHAR)) AS CPF_PAC,
        TRIM(CAST(CNS_PAC                   AS VARCHAR)) AS CNS_PAC,
        TRIM(CAST(COD_SIGTAP_PROCEDIMENTO   AS VARCHAR)) AS COD_SIGTAP_PROCEDIMENTO,
        TRIM(CAST(CBO                       AS VARCHAR)) AS CBO,
        TRIM(CAST(UF_DESC_ATEND             AS VARCHAR)) AS UF_DESC_ATEND,
        TRIM(CAST(IBGE_ATEND                AS VARCHAR)) AS IBGE_ATEND,
        TRIM(CAST(CNES_ATEND                AS VARCHAR)) AS CNES_ATEND,
        TRIM(CAST(DT_CMP_FORMATADA          AS VARCHAR)) AS DT_CMP_FORMATADA,
        TRY_STRPTIME(TRIM(CAST(DATA_SOLICITACAO AS VARCHAR)), '%d/%m/%Y')::DATE AS DATA_SOLICITACAO,
        TRY_STRPTIME(TRIM(CAST(DATA_AUTORIZACAO AS VARCHAR)), '%d/%m/%Y')::DATE AS DATA_AUTORIZACAO,
        TRY_STRPTIME(TRIM(CAST(DATA_INICIO      AS VARCHAR)), '%d/%m/%Y')::DATE AS DATA_INICIO,
        TRY_STRPTIME(TRIM(CAST(DATA_FINAL       AS VARCHAR)), '%d/%m/%Y')::DATE AS DATA_FINAL,
        TRIM(CAST(CO_APA_NUM                AS VARCHAR)) AS CO_APA_NUM
    FROM read_csv(
        '{CSV_PATH}',
        delim        = ';',
        quote        = '"',
        header       = true,
        all_varchar  = true,
        ignore_errors= true,
        parallel     = true
    )
)
TO '{PARQUET_PATH}'
(FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 500000);
""")

con.close()
print(f"✅ Parquet gerado: {PARQUET_PATH}")


<>:130: SyntaxWarning: invalid escape sequence '\S'
<>:131: SyntaxWarning: invalid escape sequence '\S'
<>:130: SyntaxWarning: invalid escape sequence '\S'
<>:131: SyntaxWarning: invalid escape sequence '\S'
C:\Users\Datasus\AppData\Local\Temp\ipykernel_10808\1021472288.py:130: SyntaxWarning: invalid escape sequence '\S'
  csv_path_sia = "base\SIA.csv"  # Arquivo de origem
C:\Users\Datasus\AppData\Local\Temp\ipykernel_10808\1021472288.py:131: SyntaxWarning: invalid escape sequence '\S'
  parquet_path_sia = "base\SIA.parquet"  # Nome do parquet final


Iniciando conversão de base\SIA.csv para base\SIA.parquet...
✅ Conversão SIA concluída: base\SIA.parquet


In [ ]:
"""
==============================================================================
CONVERSÃO SIH  CSV → Parquet
==============================================================================
"""
import duckdb
import os

CSV_PATH     = "base/SIH.csv"
PARQUET_PATH = "base/SIH.parquet"
DUCKDB_MEMORY = "2GB"

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Arquivo não encontrado: {CSV_PATH}")

con = duckdb.connect(database=":memory:")
con.execute(f"SET memory_limit='{DUCKDB_MEMORY}';")
con.execute("SET threads TO 4;")

print(f"Convertendo: {CSV_PATH}  →  {PARQUET_PATH}")

con.execute(f"""
COPY (
    SELECT
        TRIM(CAST(CPF_PAC                   AS VARCHAR)) AS CPF_PAC,
        TRIM(CAST(CNS_PAC                   AS VARCHAR)) AS CNS_PAC,
        TRIM(CAST(COD_SIGTAP_PROCEDIMENTO   AS VARCHAR)) AS COD_SIGTAP_PROCEDIMENTO,
        TRIM(CAST(CBO                       AS VARCHAR)) AS CBO,
        TRIM(CAST(UF_DESC_ATEND             AS VARCHAR)) AS UF_DESC_ATEND,
        TRIM(CAST(IBGE_ATEND                AS VARCHAR)) AS IBGE_ATEND,
        TRIM(CAST(CNES_ATEND                AS VARCHAR)) AS CNES_ATEND,
        TRIM(CAST(DT_CMP_FORMATADA          AS VARCHAR)) AS DT_CMP_FORMATADA,
        TRY_STRPTIME(TRIM(CAST(DATA_INICIO_INTERNACAO AS VARCHAR)), '%d/%m/%Y')::DATE AS DATA_INICIO_INTERNACAO,
        TRY_STRPTIME(TRIM(CAST(DATA_FINAL_INTERNACAO  AS VARCHAR)), '%d/%m/%Y')::DATE AS DATA_FINAL_INTERNACAO,
        TRIM(CAST(NU_AIH                    AS VARCHAR)) AS NU_AIH
    FROM read_csv(
        '{CSV_PATH}',
        delim        = ';',
        quote        = '"',
        header       = true,
        all_varchar  = true,
        ignore_errors= true,
        parallel     = true
    )
)
TO '{PARQUET_PATH}'
(FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 500000);
""")

con.close()
print(f"✅ Parquet gerado: {PARQUET_PATH}")


Iniciando conversão: base\SIH.csv
✅ Conversão concluída: base\SIH.parquet
